# Colab Training — YOLOv8n vs Faster R-CNN trên Pascal VOC 2012

**CO5085 — Deep Learning & Computer Vision Applications**

Notebook này train cả 2 model và lưu kết quả vào Google Drive.
Thời gian ước tính trên GPU T4:
- YOLOv8n 20 epoch: **~30 phút**
- Faster R-CNN 10 epoch: **~2 giờ**

In [ ]:
import subprocess, sys

# Check GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print("GPU:", result.stdout.strip())
else:
    print("WARNING: No GPU detected! Training will be slow.")
    print("Runtime > Change runtime type > GPU")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 1. Mount Google Drive (lưu checkpoint & kết quả)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/deeplearning_ass1_results'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Results will be saved to: {SAVE_DIR}")

## 2. Clone repo & install dependencies

In [ ]:
import subprocess, os

# Clone repo (thay bằng URL repo của bạn)
REPO_URL = "https://github.com/PhongNguyenTrung/hcmut-deeplearning-ass1"  # ← thay URL nếu khác

if not os.path.exists('/content/hcmut-deeplearning-ass1'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/hcmut-deeplearning-ass1'], check=True)
    print("Cloned successfully")
else:
    subprocess.run(['git', '-C', '/content/hcmut-deeplearning-ass1', 'pull'], check=True)
    print("Updated")

os.chdir('/content/hcmut-deeplearning-ass1')
print(f"Working dir: {os.getcwd()}")

In [ ]:
# Install dependencies
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'ultralytics>=8.0.0',
    'pycocotools>=2.0.6',
    'torchvision',
    'opencv-python-headless',
], check=True)
print("Dependencies installed")

## 3. Cấu hình đường dẫn

In [ ]:
import sys, os
sys.path.insert(0, '/content/hcmut-deeplearning-ass1')

# Tạo thư mục results
os.makedirs('exercise_2/results/metrics', exist_ok=True)
os.makedirs('exercise_2/results/checkpoints', exist_ok=True)
os.makedirs('exercise_2/results/plots', exist_ok=True)

DATA_DIR  = 'data/voc'
YOLO_DIR  = 'data/voc_yolo'
print("Paths configured")

## 4. Download Pascal VOC 2012 (~2GB)

Chỉ cần download 1 lần. Nếu đã có trong Drive, copy sang để tiết kiệm thời gian.

In [ ]:
import os, shutil

# Option A: copy từ Drive nếu đã có
DRIVE_VOC = f'{SAVE_DIR}/VOCdevkit'
if os.path.exists(DRIVE_VOC) and not os.path.exists('data/voc/VOCdevkit'):
    os.makedirs('data/voc', exist_ok=True)
    print("Copying VOC from Drive...")
    shutil.copytree(DRIVE_VOC, 'data/voc/VOCdevkit')
    print("Done copying from Drive")

# Option B: download mới (nếu chưa có)
from exercise_2.src.data import VOCDetectionDataset
print("Loading VOC 2012 train split...")
VOCDetectionDataset('data/voc', year='2012', image_set='train', transforms=None)
print("Loading VOC 2012 val split...")
VOCDetectionDataset('data/voc', year='2012', image_set='val', transforms=None)
print("VOC 2012 ready")

# Backup VOCdevkit to Drive for reuse
if not os.path.exists(f'{SAVE_DIR}/VOCdevkit'):
    print("Backing up VOCdevkit to Drive...")
    shutil.copytree('data/voc/VOCdevkit', f'{SAVE_DIR}/VOCdevkit')
    print("Backed up")

## 5. Train YOLOv8n

In [ ]:
from exercise_2.src.data import prepare_yolo_dataset

print("Converting VOC → YOLO format...")
yaml_path = prepare_yolo_dataset(
    voc_root='data/voc',
    output_dir='data/voc_yolo',
    splits=['train', 'val'],
)
print(f"YAML: {yaml_path}")

In [ ]:
from exercise_2.src.train import train_yolov8

YOLO_EPOCHS = 2  # ← thay đổi nếu muốn

best_pt = train_yolov8(
    data_yaml=yaml_path,
    model_size='n',
    epochs=YOLO_EPOCHS,
    imgsz=640,
    batch=16,
    project='exercise_2/results',
    name='yolov8n_voc',
    device='auto',
)
print(f"Best checkpoint: {best_pt}")

In [ ]:
import json, shutil
from ultralytics import YOLO
from exercise_2.src.evaluate import measure_fps
from exercise_2.src.utils import save_metrics_json

device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

model = YOLO(best_pt)
val_results = model.val(data=yaml_path, imgsz=640, verbose=False)
map50    = float(val_results.box.map50)
map5095  = float(val_results.box.map)

# Per-class AP@0.5 from ultralytics
from exercise_2.src.data import VOC_CLASSES
ap_per_class = {}
if hasattr(val_results.box, 'ap50') and hasattr(val_results.box, 'ap_class_index'):
    ap50_arr = val_results.box.ap50
    class_idx = val_results.box.ap_class_index.astype(int)
    for i, cls_id in enumerate(class_idx):
        if 0 <= cls_id < len(VOC_CLASSES):
            ap_per_class[VOC_CLASSES[cls_id]] = float(ap50_arr[i])
print(f'Per-class AP entries: {len(ap_per_class)}')
print(f"YOLOv8n  mAP@0.5     = {map50*100:.1f}%")
print(f"YOLOv8n  mAP@0.5:0.95= {map5095*100:.1f}%")

fps_info = measure_fps(model, 'yolo', device=device)
print(f"FPS ({device}): {fps_info['fps']} | {fps_info['ms_per_image']} ms/img")

yolo_results = {
    'model': 'YOLOv8n', 'type': 'one-stage',
    'dataset': 'Pascal VOC 2012',
    'mAP_50': map50, 'mAP_50_95': map5095,
    'AP_per_class': ap_per_class,
    'fps': fps_info['fps'], 'ms_per_image': fps_info['ms_per_image'],
    'fps_device': fps_info['device'], 'epochs': YOLO_EPOCHS,
    'checkpoint': best_pt,
}
save_metrics_json(yolo_results, 'exercise_2/results/metrics/yolo_results.json')
shutil.copy('exercise_2/results/metrics/yolo_results.json', f'{SAVE_DIR}/yolo_results.json')
print("Saved yolo_results.json")

## 6. Train Faster R-CNN ResNet-50 FPN

In [ ]:
from exercise_2.src.data import get_frcnn_loaders
from exercise_2.src.models import get_faster_rcnn, get_model_info
from exercise_2.src.train import fit_frcnn, load_frcnn_checkpoint

FRCNN_EPOCHS = 2  # ← thay đổi nếu muốn
FRCNN_CKPT   = 'exercise_2/results/checkpoints/frcnn_voc.pth'

train_loader, val_loader = get_frcnn_loaders('data/voc', batch_size=4, num_workers=2)
model = get_faster_rcnn(num_classes=21)
info  = get_model_info(model, 'Faster R-CNN')
print(f"Params: {info['total_params']} | Trainable: {info['trainable_params']}")

history = fit_frcnn(model, train_loader, val_loader, config={
    'epochs':    FRCNN_EPOCHS,
    'lr':        0.005,
    'device':    'cuda' if __import__('torch').cuda.is_available() else 'cpu',
    'save_path': FRCNN_CKPT,
})
import shutil
shutil.copy(FRCNN_CKPT, f'{SAVE_DIR}/frcnn_voc.pth')
print(f"Checkpoint backed up to Drive")

In [ ]:
from exercise_2.src.train import load_frcnn_checkpoint
from exercise_2.src.evaluate import predict_frcnn, compute_map_coco, measure_fps
from exercise_2.src.utils import save_metrics_json, plot_loss_curves

device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
model  = load_frcnn_checkpoint(model, FRCNN_CKPT, device)

print("Running inference on val set...")
preds, targets = predict_frcnn(model, val_loader, device, score_threshold=0.05)
map_results = compute_map_coco(preds, targets)
print(f"Faster R-CNN  mAP@0.5     = {map_results['mAP_50']*100:.1f}%")
print(f"Faster R-CNN  mAP@0.5:0.95= {map_results['mAP_50_95']*100:.1f}%")

fps_info = measure_fps(model, 'frcnn', device=device)
print(f"FPS ({device}): {fps_info['fps']} | {fps_info['ms_per_image']} ms/img")

plot_loss_curves(history, 'Faster R-CNN Training Loss', 'exercise_2/results/plots/frcnn_loss.png')

frcnn_results = {
    'model': 'Faster R-CNN ResNet-50 FPN', 'type': 'two-stage',
    'dataset': 'Pascal VOC 2012',
    'mAP_50': map_results['mAP_50'], 'mAP_50_95': map_results['mAP_50_95'],
    'AP_per_class': map_results.get('AP_per_class', {}),
    'fps': fps_info['fps'], 'ms_per_image': fps_info['ms_per_image'],
    'fps_device': fps_info['device'], 'epochs': FRCNN_EPOCHS,
    'params': info['total_params'], 'checkpoint': FRCNN_CKPT,
    'history': history,
}
save_metrics_json(frcnn_results, 'exercise_2/results/metrics/frcnn_results.json')
import shutil
shutil.copy('exercise_2/results/metrics/frcnn_results.json', f'{SAVE_DIR}/frcnn_results.json')
print("Saved frcnn_results.json")

## 7. So sánh & sinh plots

In [ ]:
from exercise_2.src.utils import (
    plot_map_comparison, plot_speed_accuracy_tradeoff,
    plot_per_class_ap, print_detection_results_table,
    load_metrics_json,
)
import shutil

yolo_r  = load_metrics_json('exercise_2/results/metrics/yolo_results.json')
frcnn_r = load_metrics_json('exercise_2/results/metrics/frcnn_results.json')
all_results = {'YOLOv8n': yolo_r, 'Faster R-CNN': frcnn_r}

print_detection_results_table(all_results)

plot_map_comparison(all_results, 'exercise_2/results/plots/map_comparison.png')
plot_speed_accuracy_tradeoff(all_results, 'exercise_2/results/plots/speed_accuracy.png')
if frcnn_r.get('AP_per_class'):
    plot_per_class_ap(all_results, 'exercise_2/results/plots/per_class_ap.png')

# Backup plots to Drive
for f in ['map_comparison.png', 'speed_accuracy.png', 'frcnn_loss.png', 'per_class_ap.png']:
    src = f'exercise_2/results/plots/{f}'
    if __import__('os').path.exists(src):
        shutil.copy(src, f'{SAVE_DIR}/{f}')

print(f"\nAll results backed up to {SAVE_DIR}")
print("Download từ Google Drive hoặc:")
print("  from google.colab import files")
print("  files.download('exercise_2/results/metrics/yolo_results.json')")
print("  files.download('exercise_2/results/metrics/frcnn_results.json')")

## 8. Download kết quả về máy

In [ ]:
from google.colab import files
import os

for f in [
    'exercise_2/results/metrics/yolo_results.json',
    'exercise_2/results/metrics/frcnn_results.json',
    'exercise_2/results/plots/map_comparison.png',
    'exercise_2/results/plots/speed_accuracy.png',
]:
    if os.path.exists(f):
        files.download(f)
        print(f"Downloaded: {f}")